# 🔍 Explainability & Clinical Baseline Comparison
## SHAP Analysis + Value Added of Genomic Features

**Purpose:**
1. Interpret the trained model using SHAP values
2. Compare genomic model vs clinical baseline
3. Quantify value added by genomic features
4. Statistical comparison (DeLong test)

---

In [ ]:
# Setup
import sys
sys.path.append('/workspace/core')

import pandas as pd
import numpy as np
import pickle
import json
import logging
from src.io import setup_logging, logger
from src.data_loaders import load_tcga_prad_bcr
from src.improved_pipeline import CLINICAL_BASELINE_FEATURES, evaluate_clinical_baseline
from src.explainability import plot_shap_summary, plot_shap_beeswarm
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

setup_logging(level=logging.INFO)
RANDOM_STATE = 42

## Step 1: Load Model and Data

In [ ]:
# Load trained model
with open('/workspace/core/models/final_bcr_model.pkl', 'rb') as f:
    model_data = pickle.load(f)

final_model = model_data['model']
trained_features = model_data['features']
train_auc = model_data['genomic_auc']
clinical_auc = model_data['clinical_auc']

# Load data
clinical, rna_seq, bcr_labels = load_tcga_prad_bcr()
merged = clinical.merge(rna_seq, left_index=True, right_index=True, how='inner')
X = merged.drop(columns=[c for c in ['BCR', 'days_to_bcr'] if c in merged.columns], errors='ignore')
y = bcr_labels.reindex(X.index).dropna()
X = X.loc[y.index]

# Preprocess
X_clean = X.dropna(axis=1, thresh=len(X)*0.8)
X_clean = X_clean.fillna(X_clean.median())
gene_cols = [c for c in X_clean.columns if c not in CLINICAL_BASELINE_FEATURES]
X_clean[gene_cols] = np.log2(X_clean[gene_cols] + 1)

logger.info(f"Data loaded: {X_clean.shape[0]} samples, {len(trained_features)} features")

## Step 2: SHAP Explainability

In [ ]:
# Calculate SHAP values
try:
    import shap
    
    # Create explainer for calibrated model
    explainer = shap.Explainer(final_model.predict_proba, X_clean[trained_features])
    shap_values = explainer(X_clean[trained_features])
    
    # Summary plot
    fig1 = plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, X_clean[trained_features], show=False)
    plt.title('SHAP Summary Plot - Genomic Model')
    plt.tight_layout()
    plt.savefig('/workspace/core/results/shap_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Beeswarm plot
    fig2 = plt.figure(figsize=(10, 6))
    shap.beeswarm_plot(shap_values, show=False)
    plt.title('SHAP Beeswarm Plot')
    plt.tight_layout()
    plt.savefig('/workspace/core/results/shap_beeswarm.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    logger.info("SHAP plots saved to /workspace/core/results/")
    
    # Top features by mean |SHAP|
    mean_shap = np.abs(shap_values.values).mean(axis=0)
    top_features = pd.Series(mean_shap, index=trained_features).sort_values(ascending=False)
    
    print("\nTop 10 Features by SHAP Importance:")
    print(top_features.head(10))
    
except ImportError:
    logger.warning("SHAP not installed. Install with: pip install shap")
    top_features = None

## Step 3: Clinical Baseline Evaluation

In [ ]:
# Evaluate clinical baseline on same data
clinical_results = evaluate_clinical_baseline(X_clean, y, random_state=RANDOM_STATE)

print("\n" + "="*60)
print("📊 CLINICAL BASELINE (Reference)")
print("="*60)
print(f"Features: {clinical_results['n_features']}")
print(f"AUC: {clinical_results['clinical_auc_mean']:.3f} ± {clinical_results['clinical_auc_std']:.3f}")
print(f"Features used: {CLINICAL_BASELINE_FEATURES}")
print("="*60)

## Step 4: Combined Model (Clinical + Genomic)

In [ ]:
# Train combined model
from src.improved_pipeline import build_elastic_net, stability_selection
from sklearn.model_selection import cross_val_score

# Combine clinical + genomic features
combined_features = list(set(CLINICAL_BASELINE_FEATURES) & set(X_clean.columns)) + trained_features
combined_features = list(set(combined_features))  # Remove duplicates

logger.info(f"Combined model: {len(combined_features)} features")

# Cross-validation for combined model
combined_model = build_elastic_net(random_state=RANDOM_STATE)
cv_scores = cross_val_score(
    combined_model, X_clean[combined_features], y,
    cv=5, scoring='roc_auc'
)

combined_auc = cv_scores.mean()
combined_auc_std = cv_scores.std()

print("\n" + "="*60)
print("🧬 COMBINED MODEL (Clinical + Genomic)")
print("="*60)
print(f"Features: {len(combined_features)}")
print(f"AUC: {combined_auc:.3f} ± {combined_auc_std:.3f}")
print("="*60)

## Step 5: Statistical Comparison (DeLong Test)

In [ ]:
# DeLong test for AUC comparison
try:
    from scipy import stats
    
    # Get predictions from both models
    clinical_model = build_elastic_net(random_state=RANDOM_STATE)
    clinical_cols = [c for c in CLINICAL_BASELINE_FEATURES if c in X_clean.columns]
    clinical_model.fit(X_clean[clinical_cols], y)
    
    prob_clinical = clinical_model.predict_proba(X_clean[clinical_cols])[:, 1]
    prob_combined = combined_model.fit(X_clean[combined_features], y).predict_proba(X_clean[combined_features])[:, 1]
    
    # Simple z-test for AUC difference (approximation)
    auc_clinical = roc_auc_score(y, prob_clinical)
    auc_combined = roc_auc_score(y, prob_combined)
    
    # Bootstrap for confidence interval
    n_boot = 1000
    auc_diffs = []
    rng = np.random.RandomState(RANDOM_STATE)
    
    for _ in range(n_boot):
        idx = rng.choice(len(y), size=len(y), replace=True)
        diff = roc_auc_score(y.iloc[idx], prob_combined[idx]) - roc_auc_score(y.iloc[idx], prob_clinical[idx])
        auc_diffs.append(diff)
    
    ci_lower = np.percentile(auc_diffs, 2.5)
    ci_upper = np.percentile(auc_diffs, 97.5)
    p_value = (np.sum(np.array(auc_diffs) <= 0) + 1) / (n_boot + 1)
    
    print("\n" + "="*60)
    print("📈 STATISTICAL COMPARISON (Bootstrap DeLong-style)")
    print("="*60)
    print(f"Clinical AUC:      {auc_clinical:.3f}")
    print(f"Combined AUC:      {auc_combined:.3f}")
    print(f"Difference:        {auc_combined - auc_clinical:+.3f}")
    print(f"95% CI:            [{ci_lower:.3f}, {ci_upper:.3f}]")
    print(f"P-value:           {p_value:.4f}")
    
    if p_value < 0.05:
        print("\n✅ Significant improvement (p < 0.05)")
    else:
        print("\n⚠️ No significant improvement (p >= 0.05)")
    print("="*60)
    
except Exception as e:
    logger.error(f"Statistical test failed: {e}")

## Step 6: Value Added Analysis

In [ ]:
# Calculate value added by genomic features
value_added = auc_combined - auc_clinical
relative_improvement = (auc_combined - auc_clinical) / auc_clinical * 100

print("\n" + "="*60)
print("💎 VALUE ADDED BY GENOMIC FEATURES")
print("="*60)
print(f"Absolute improvement: {value_added:+.3f} AUC points")
print(f"Relative improvement: {relative_improvement:+.1f}%")
print(f"NNT* (Number Needed to Test): {1/abs(value_added):.1f} patients")
print("="*60)
print("\n*NNT: How many patients need genomic testing to prevent one misclassification")

# Save results
explainability_results = {
    'clinical_auc': auc_clinical,
    'combined_auc': auc_combined,
    'value_added_absolute': value_added,
    'value_added_relative_percent': relative_improvement,
    'bootstrap_ci_95': [ci_lower, ci_upper],
    'p_value': p_value,
    'top_shap_features': top_features.head(10).to_dict() if top_features is not None else None
}

with open('/workspace/core/results/explainability_comparison.json', 'w') as f:
    json.dump(explainability_results, f, indent=2)

logger.info("Results saved to /workspace/core/results/explainability_comparison.json")

## Summary

✅ **Completed:**
- SHAP explainability analysis
- Clinical baseline evaluation
- Combined model training
- Statistical comparison (DeLong-style bootstrap)
- Value added quantification

📌 **Key Findings:**
- Genomic features add **{value_added:.3f}** AUC points over clinical alone
- Top predictive genes identified by SHAP
- Statistical significance: **p = {p_value:.4f}**